In [1]:
%load_ext autoreload
%autoreload 2
import time

import numpy as np
from PIL import Image
from paddleocr import PaddleOCR

from fraude_recibos import config

FINDIT_DIR = config.FINDIT_DIR


def cargar_canonica(ruta):
    """Representación única para todo el proyecto: escala de grises, sin alfa."""
    with Image.open(FINDIT_DIR / ruta) as img:
        if img.mode in ("RGBA", "LA"):
            fondo = Image.new("RGBA", img.size, (255, 255, 255, 255))
            img = Image.alpha_composite(fondo, img.convert("RGBA"))
        return np.array(img.convert("L"))


def a_tres_canales(gris):
    """PaddleOCR espera 3 canales: se replica el gris para no reintroducir color."""
    return np.stack([gris, gris, gris], axis=-1)


def crear_ocr(modelo_det, limite=None):
    opciones = dict(
        lang="en",
        text_detection_model_name=modelo_det,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    if limite:
        opciones.update(text_det_limit_type="max", text_det_limit_side_len=limite)
    return PaddleOCR(**opciones)


imagen = a_tres_canales(cargar_canonica("train/X51005230621.png"))
print("Tamaño de la imagen:", imagen.shape[:2], "\n")

configuraciones = {
    "server_sin_limite": crear_ocr("PP-OCRv5_server_det"),
    "server_1280": crear_ocr("PP-OCRv5_server_det", limite=1280),
    "mobile_1280": crear_ocr("PP-OCRv5_mobile_det", limite=1280),
}

for nombre, motor in configuraciones.items():
    motor.predict(imagen)  # calentamiento: la primera ejecución siempre es más lenta
    inicio = time.perf_counter()
    resultado = motor.predict(imagen)[0]
    duracion = time.perf_counter() - inicio
    print(f"{nombre:<20} {duracion:6.1f} s | {len(resultado['rec_texts'])} líneas")

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Tamaño de la imagen: (1769, 593) 



/var/folders/bq/2p06267d7z9cgzh3lj49qf4r0000gn/T/ipykernel_73964/1534030876.py:38: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  return PaddleOCR(**opciones)
/Users/jmarquez/fraude-recibos/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_server_rec`.
/var/fold

server_sin_limite     101.2 s | 59 líneas
server_1280            13.6 s | 59 líneas
mobile_1280             7.1 s | 51 líneas


In [2]:
#Celda de lectura de etiquetas y cobertura
import json
import pandas as pd

SROIE_DIR = config.RAW_DIR / "sroie"


def cargar_etiquetas_sroie():
    """Etiquetas de SROIE desde train y test: para nosotros es solo un diccionario."""
    filas = []
    for split_sroie in ["train", "test"]:
        for archivo in (SROIE_DIR / split_sroie / "entities").glob("*.txt"):
            etiquetas = json.loads(archivo.read_text(encoding="utf-8"))
            filas.append({"id": archivo.stem, **etiquetas})
    return pd.DataFrame(filas).drop_duplicates("id")


etiquetas = cargar_etiquetas_sroie()
metadata = pd.read_parquet(config.PROCESSED_DIR / "metadata.parquet")

# Recibos de evaluación del OCR: auténticos de entrenamiento
evaluacion = metadata[(metadata["split"] == "train") & (metadata["forged"] == 0)]
evaluacion = evaluacion.merge(etiquetas, on="id", how="left")

print("Etiquetas SROIE cargadas:", len(etiquetas))
print("Recibos auténticos de entrenamiento:", len(evaluacion))
print("Sin etiquetas SROIE:", evaluacion["total"].isna().sum())

# Los recibos sin etiqueta no se pueden evaluar: se excluyen
print("\nRecibos sin etiqueta (excluidos de la evaluación):")
print(evaluacion.loc[evaluacion["total"].isna(), "id"].tolist())

evaluacion = evaluacion.dropna(subset=["total"]).reset_index(drop=True)
print("Recibos para evaluar el OCR:", len(evaluacion))

evaluacion[["id", "company", "date", "total"]].head()

Etiquetas SROIE cargadas: 973
Recibos auténticos de entrenamiento: 481
Sin etiquetas SROIE: 6

Recibos sin etiqueta (excluidos de la evaluación):
['X51005705759', 'X51006401940', 'X51006414592', 'X51006414632', 'X51006502535', 'X51006913067']
Recibos para evaluar el OCR: 475


,id,company,date,total
0,X00016469623,MR D.I.Y. (M) SDN BHD,18-11-18,30.90
1,X00016469670,OJC MARKETING SDN BHD,15/01/2019,193.00
2,X00016469671,OJC MARKETING SDN BHD,02/01/2019,170.00
3,X00016469672,SOON HUAT MACHINERY ENTERPRISE,11/01/2019,327.00
4,X51005200938,PERNIAGAAN ZHENG HUI,12/02/2018,112.45


In [3]:
#Paso 2: Celda de configuración
import logging
import re
from difflib import SequenceMatcher

# Silenciar los mensajes de descarga de modelos
logging.getLogger("httpx").setLevel(logging.WARNING)

CONFIGURACIONES = {
    "A_server_server": ("PP-OCRv5_server_det", "PP-OCRv5_server_rec"),
    "B_mobile_server": ("PP-OCRv5_mobile_det", "PP-OCRv5_server_rec"),
    "C_mobile_mobile_en": ("PP-OCRv5_mobile_det", "en_PP-OCRv5_mobile_rec"),
}
LIMITE_LADO = 1280


def crear_ocr(modelo_det, modelo_rec):
    return PaddleOCR(
        text_detection_model_name=modelo_det,
        text_recognition_model_name=modelo_rec,
        text_det_limit_type="max",
        text_det_limit_side_len=LIMITE_LADO,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

In [4]:
#Paso 3: Celda de caché
def ruta_cache(nombre_config, id_recibo):
    return config.INTERIM_DIR / "ocr" / nombre_config / f"{id_recibo}.json"


def ejecutar_ocr(motor, nombre_config, fila):
    """Ejecuta el OCR una sola vez por recibo y configuración; luego lee del disco."""
    destino = ruta_cache(nombre_config, fila.id)
    if destino.exists():
        return json.loads(destino.read_text(encoding="utf-8"))

    imagen = a_tres_canales(cargar_canonica(fila.ruta_imagen))
    inicio = time.perf_counter()
    resultado = motor.predict(imagen)[0]
    segundos = time.perf_counter() - inicio

    datos = {
        "id": fila.id,
        "segundos": segundos,
        "textos": list(resultado["rec_texts"]),
        "scores": [float(s) for s in resultado["rec_scores"]],
        "cajas": np.asarray(resultado["rec_boxes"]).tolist(),
    }
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(json.dumps(datos, ensure_ascii=False), encoding="utf-8")
    return datos

In [5]:
#Paso 4: Celda de normalización y comparación

PATRON_MONTO = re.compile(r"\d{1,3}(?:,\d{3})+\.\d{2}|\d+\.\d{2}|\d+,\d{2}(?!\d)")
MESES = {m: i for i, m in enumerate(
    ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"], 1)}


def montos_en(texto):
    """Todos los montos con dos decimales, como números."""
    montos = set()
    for m in PATRON_MONTO.findall(texto):
        if "." in m:
            m = m.replace(",", "")
        else:
            m = m.replace(",", ".")
        montos.add(round(float(m), 2))
    return montos


def monto_etiqueta(valor):
    limpio = re.sub(r"[^\d.]", "", str(valor))
    return round(float(limpio), 2) if limpio else None


def anio_completo(a):
    a = int(a)
    return a + 2000 if a < 100 else a


def fechas_en(texto):
    """Todas las fechas como tuplas (año, mes, día), sin importar el formato."""
    texto = texto.upper()
    fechas = set()
    for d, m, a in re.findall(r"(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{2,4})", texto):
        fechas.add((anio_completo(a), int(m), int(d)))
    for a, m, d in re.findall(r"(\d{4})[/\-.](\d{1,2})[/\-.](\d{1,2})", texto):
        fechas.add((int(a), int(m), int(d)))
    for d, mes, a in re.findall(r"(\d{1,2})[\s\-/.]*([A-Z]{3})[A-Z]*[\s\-/.,]*(\d{2,4})", texto):
        if mes in MESES:
            fechas.add((anio_completo(a), MESES[mes], int(d)))
    return fechas


def normalizar(texto):
    texto = re.sub(r"[^A-Z0-9 ]", " ", str(texto).upper())
    return re.sub(r"\s+", " ", texto).strip()


def similitud_empresa(etiqueta, lineas):
    """Mejor similitud entre el nombre etiquetado y cualquier línea (o par de líneas)."""
    objetivo = normalizar(etiqueta)
    candidatos = lineas + [a + " " + b for a, b in zip(lineas, lineas[1:])]
    if not objetivo or not candidatos:
        return 0.0
    return max(SequenceMatcher(None, objetivo, normalizar(c)).ratio() for c in candidatos)


# Prueba rápida con el recibo de la prueba de humo
print(montos_en("TOTAL: 7.30  CASH 1,250.00"))
print(fechas_en("Date: 05/02/2018") == fechas_en("05-02-18"))
print(round(similitud_empresa("SIN LIANHAP SDN BHD", ["SIN LIanHAP SON BhD"]), 2))
print(fechas_en("05-AUG-2017") == fechas_en("05/08/2017"))

{1250.0, 7.3}
True
0.95
True


In [6]:
#Paso 5: Celda de ejecución sobre la muestra
ids_muestra = pd.read_csv(config.REPORTS_DIR / "muestra_ocr_ids.csv")["id"]
muestra = evaluacion[evaluacion["id"].isin(ids_muestra)].reset_index(drop=True)
print("Recibos en la muestra:", len(muestra))

for nombre, (det, rec) in CONFIGURACIONES.items():
    print(f"Procesando {nombre}...")
    motor = crear_ocr(det, rec)
    motor.predict(a_tres_canales(cargar_canonica(muestra.iloc[0].ruta_imagen)))  # calentamiento
    for i, fila in enumerate(muestra.itertuples(), 1):
        ejecutar_ocr(motor, nombre, fila)
        if i % 10 == 0:
            print(f"  {i}/{len(muestra)}")
    del motor

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_server_det`.


Recibos en la muestra: 48
Procesando A_server_server...


Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_server_rec`.
Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_server_rec`.


  10/48
  20/48
  30/48
  40/48
Procesando B_mobile_server...


Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/en_PP-OCRv5_mobile_rec`.


  10/48
  20/48
  30/48
  40/48
Procesando C_mobile_mobile_en...
  10/48
  20/48
  30/48
  40/48


In [7]:
#Paso 6: Celda de resultados
filas = []
for nombre in CONFIGURACIONES:
    for fila in muestra.itertuples():
        datos = json.loads(ruta_cache(nombre, fila.id).read_text(encoding="utf-8"))
        texto = "\n".join(datos["textos"])
        filas.append({
            "config": nombre,
            "id": fila.id,
            "total_ok": monto_etiqueta(fila.total) in montos_en(texto),
            "fecha_ok": bool(fechas_en(str(fila.date)) & fechas_en(texto)),
            "sim_empresa": similitud_empresa(fila.company, datos["textos"]),
            "segundos": datos["segundos"],
        })

comparacion = pd.DataFrame(filas)
comparacion["empresa_ok"] = comparacion["sim_empresa"] >= 0.8

resumen = comparacion.groupby("config").agg(
    total=("total_ok", "mean"),
    fecha=("fecha_ok", "mean"),
    empresa=("empresa_ok", "mean"),
    seg_mediana=("segundos", "median"),
    seg_max=("segundos", "max"),
)
resumen[["total", "fecha", "empresa"]] = (resumen[["total", "fecha", "empresa"]] * 100).round(1)
resumen.round(1)

,total,fecha,empresa,seg_mediana,seg_max
config,,,,,
A_server_server,100.0,89.6,85.4,65.8,102.3
B_mobile_server,100.0,87.5,87.5,5.6,10.1
C_mobile_mobile_en,100.0,100.0,87.5,7.6,12.5


In [8]:
#Paso 7: Verificación pareada y análisis de errores
pivote = comparacion.pivot(index="id", columns="config", values="fecha_ok")
solo_c = (pivote["C_mobile_mobile_en"] & ~pivote["B_mobile_server"]).sum()
solo_b = (pivote["B_mobile_server"] & ~pivote["C_mobile_mobile_en"]).sum()
print(f"Fechas: C acierta y B falla en {solo_c} recibos; B acierta y C falla en {solo_b}.\n")

# Errores de la configuración elegida
errores_c = comparacion[comparacion["config"] == "C_mobile_mobile_en"].merge(
    muestra[["id", "company", "date", "total"]], on="id"
)

for campo, columna in [("total", "total_ok"), ("fecha", "fecha_ok"), ("empresa", "empresa_ok")]:
    fallos = errores_c[~errores_c[columna]]
    print(f"--- {campo.upper()}: {len(fallos)} fallos ---")
    for fila in fallos.itertuples():
        datos = json.loads(ruta_cache("C_mobile_mobile_en", fila.id).read_text(encoding="utf-8"))
        if campo == "total":
            detalle = f"esperado {fila.total} | montos leídos: {sorted(montos_en(chr(10).join(datos['textos'])))[-5:]}"
        elif campo == "fecha":
            candidatas = [t for t in datos["textos"] if re.search(r"\d{1,4}[/\-.]\d{1,2}", t)][:3]
            detalle = f"esperado {fila.date} | líneas con forma de fecha: {candidatas}"
        else:
            detalle = f"esperado '{fila.company}' | similitud {fila.sim_empresa:.2f}"
        print(f"{fila.id}: {detalle}")
    print()

Fechas: C acierta y B falla en 6 recibos; B acierta y C falla en 0.

--- TOTAL: 0 fallos ---

--- FECHA: 0 fallos ---

--- EMPRESA: 6 fallos ---
X51005337867: esperado 'OLD TOWN KOPITAM SND BHD' | similitud 0.79
X51005757201: esperado 'AEON CO. (M) BHD.' | similitud 0.74
X51006387971: esperado '99 SPEED MART S/B' | similitud 0.65
X51006466066: esperado 'MENTAI INITIAL SDN BHD' | similitud 0.78
X51006619757: esperado '99 SPEED MART S/B' | similitud 0.79
X51008164420: esperado 'POPULAR BOOK CO. (M) SDN BHD' | similitud 0.75



muestra[["id"]].to_csv(config.REPORTS_DIR / "muestra_ocr_ids.csv", index=False)
print("Muestra guardada:", len(muestra), "recibos")

In [9]:
print("\nRecibos sin etiqueta (excluidos de la evaluación):")
print(evaluacion.loc[evaluacion["total"].isna(), "id"].tolist())

evaluacion = evaluacion.dropna(subset=["total"]).reset_index(drop=True)
print("Recibos para evaluar el OCR:", len(evaluacion))


Recibos sin etiqueta (excluidos de la evaluación):
[]
Recibos para evaluar el OCR: 475


In [10]:
CONFIG_ELEGIDA = "C_mobile_mobile_en"

motor = crear_ocr(*CONFIGURACIONES[CONFIG_ELEGIDA])
motor.predict(a_tres_canales(cargar_canonica(evaluacion.iloc[0].ruta_imagen)))  # calentamiento

inicio_total = time.perf_counter()
for i, fila in enumerate(evaluacion.itertuples(), 1):
    ejecutar_ocr(motor, CONFIG_ELEGIDA, fila)
    if i % 25 == 0:
        minutos = (time.perf_counter() - inicio_total) / 60
        print(f"{i}/{len(evaluacion)}  ({minutos:.1f} min)")
del motor
print("Listo.")

Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/jmarquez/.paddlex/official_models/en_PP-OCRv5_mobile_rec`.


KeyboardInterrupt: 